# Notebook 04 — Power Analysis & Sample Size
**Tujuan:** Demonstrasikan experiment design thinking — berapa sampel & berapa lama eksperimen harus berjalan?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import norm
from statsmodels.stats.power import NormalIndPower, TTestIndPower
from statsmodels.stats.proportion import proportion_effectsize
from pathlib import Path

OUT     = Path('../output')
FIGURES = OUT / 'figures'

BLUE  = '#2563EB'
RED   = '#DC2626'
GREEN = '#059669'
GRAY  = '#6B7280'
LIGHT = '#93C5FD'

df = pd.read_parquet(OUT / 'df_clean.parquet')
p_ctrl  = df[df['group']=='control']['converted'].mean()
p_treat = df[df['group']=='treatment']['converted'].mean()
daily_traffic = df.groupby(df['timestamp'].dt.date)['user_id'].count().mean()

print(f'Baseline conversion rate : {p_ctrl:.4f}')
print(f'Observed treatment rate  : {p_treat:.4f}')
print(f'Daily traffic            : {daily_traffic:,.0f} users/hari')

## A. Effect Size Aktual (Cohen's h)

In [ ]:
# Cohen's h untuk perbedaan proporsi
h_actual = proportion_effectsize(p_treat, p_ctrl)

print(f"Cohen's h (aktual) : {h_actual:.4f}")
print(f"  → Interpretasi   : {'Kecil (<0.2)' if abs(h_actual) < 0.2 else 'Sedang (0.2-0.5)' if abs(h_actual) < 0.5 else 'Besar (>0.5)'}")

## B. Power Curve — Sample Size vs Statistical Power

In [ ]:
analysis = NormalIndPower()
sample_sizes = np.arange(500, 150_001, 500)

# Hitung power untuk berbagai effect size
effect_sizes = {
    'Kecil (h=0.05)':  0.05,
    'Aktual (h={:.3f})'.format(abs(h_actual)): abs(h_actual),
    'Sedang (h=0.20)':  0.20,
    'Besar  (h=0.50)':  0.50,
}
colors_es = [RED, BLUE, GREEN, GRAY]

fig, ax = plt.subplots(figsize=(13, 6))

for (label, h), color in zip(effect_sizes.items(), colors_es):
    powers = [analysis.power(effect_size=h, nobs1=n, alpha=0.05, ratio=1.0)
              for n in sample_sizes]
    ax.plot(sample_sizes / 1000, powers, color=color, linewidth=2, label=label)

ax.axhline(0.80, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Power = 80% (standar)')
ax.axhline(0.90, color='black', linestyle=':',  linewidth=1.5, alpha=0.5, label='Power = 90%')

# Annotasi: sample size untuk 80% power dengan effect aktual
n_80 = analysis.solve_power(effect_size=abs(h_actual), power=0.80, alpha=0.05, ratio=1.0)
ax.axvline(n_80 / 1000, color=BLUE, linestyle='-.', linewidth=1.5, alpha=0.8)
ax.text(n_80 / 1000 + 0.5, 0.35,
        f'n={n_80:,.0f}\nper grup\n(80% power)', color=BLUE, fontsize=9)

ax.set_xlabel('Sample Size per Grup (ribu)')
ax.set_ylabel('Statistical Power')
ax.set_title('Power Curve — Sample Size vs Power\n(α=0.05, two-sided)', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim(0, 150)

plt.tight_layout()
plt.savefig(FIGURES / 'E_power_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nUntuk 80% power dengan effect aktual: butuh {n_80:,.0f} user per grup')
print('Saved: E_power_curve.png')

## C. Tabel Panduan: MDE → Sample Size → Days to Run

In [ ]:
mde_targets = [0.005, 0.010, 0.020, 0.030, 0.050]
rows = []

for mde in mde_targets:
    p2 = p_ctrl + mde  # target conversion rate treatment
    if p2 >= 1:
        continue
    h  = proportion_effectsize(p2, p_ctrl)
    n  = analysis.solve_power(effect_size=abs(h), power=0.80, alpha=0.05, ratio=1.0)
    total_n = n * 2  # kedua grup
    days    = total_n / daily_traffic
    rows.append({
        'MDE (Absolute)': f'+{mde:.1%}',
        'Target Conv Rate': f'{p2:.3f}',
        "Cohen's h": round(h, 4),
        'Sample/Grup': f'{n:,.0f}',
        'Total Sample': f'{total_n:,.0f}',
        'Days to Run': f'{days:.0f} hari'
    })

guide_df = pd.DataFrame(rows)
print('=== PANDUAN EKSPERIMEN ===')
print(f'Baseline conversion rate : {p_ctrl:.4f}')
print(f'Daily traffic (avg)      : {daily_traffic:,.0f} users/hari')
print(f'Target power             : 80% | α = 0.05')
print()
print(guide_df.to_string(index=False))

## D. Novelty Effect Check

In [ ]:
df['date'] = df['timestamp'].dt.date
daily_conv = (
    df.groupby(['date','group'])['converted']
    .mean()
    .reset_index()
)

# Tambahkan day_number relatif dari awal eksperimen
start_date = daily_conv['date'].min()
daily_conv['day_num'] = (pd.to_datetime(daily_conv['date']) - pd.to_datetime(start_date)).dt.days + 1

fig, ax = plt.subplots(figsize=(13, 5))

for grp, color in [('control', BLUE), ('treatment', RED)]:
    d = daily_conv[daily_conv['group'] == grp].sort_values('day_num')
    ax.plot(d['day_num'], d['converted'] * 100,
            color=color, linewidth=1.5, alpha=0.5, label=f'_daily_{grp}')
    # Rolling 7-day average
    rolling = d.set_index('day_num')['converted'].rolling(7, min_periods=3).mean()
    ax.plot(rolling.index, rolling.values * 100,
            color=color, linewidth=2.5, label=f'{grp} (7-day avg)')

ax.set_xlabel('Hari ke-N Eksperimen')
ax.set_ylabel('Conversion Rate (%)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title('Novelty Effect Check — Conversion Rate Harian\n'
             '(jika treatment turun setelah hari-hari awal → novelty effect terdeteksi)',
             fontweight='bold')
ax.legend()
ax.axvline(7, color=GRAY, linestyle='--', linewidth=1, alpha=0.5, label='_Minggu 1')
ax.axvline(14, color=GRAY, linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES / 'F_novelty_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: F_novelty_effect.png')